In [ ]:
from model import *
from utils import *

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt 
from PIL import Image 
import os 

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

train_data = load_data(r'\train\train\train.npz')
X, y, _ = prepare_train_data(train_data)

DATASET_SIZE = len(X)
# Split train/val
# This is done only to measure generalization capabilities, you don't have to
# use a validation set (though we encourage this)
n_train = int(0.9 * len(X))
TRAIN_SPLIT = torch.zeros(len(X), dtype=torch.bool)
TRAIN_SPLIT[:n_train] = 1
X_train, X_val = X[TRAIN_SPLIT], X[~TRAIN_SPLIT]
y_train, y_val = y[TRAIN_SPLIT], y[~TRAIN_SPLIT]


train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256)

cuda
(125000,)
Train data: 125000 captions, 125000 images


# Ensemble Model  

The next code is the optuna grid search that is used to save the top 8 models, which will be used in our Ensemble Model. The code can be runned, but there is no seed that ensure the same output models, sorry :( . The best 8 models used for the submission are saved in the Models/Best_submission folder, and it is possible to import them running the next cell. At the and of the output of the optuna cell, are printed the best models with their mrr score, and the combination of learning rate and temperature used for the training.

In [ ]:
import optuna
import torch
import numpy as np
import os


TOP_K = 8  # Number of top models to save


best_model = None
best_mrr = -np.inf

study = optuna.create_study(
    study_name="real_final_1408_topk",
    direction="maximize",
    storage="sqlite:///optuna_study_real_final_1408_davvero_2.db", # scusate per il nome, avevo finito la fantasia
    load_if_exists=True
)

study.optimize(objective, n_trials=45, gc_after_trial=True)

# Results summary

best_trial = study.best_trial
print("\n" + "="*60)
print("MIGLIOR MODELLO (per MRR)")
print("="*60)
print(f"MRR: {best_trial.value:.4f}")
print(f"Top1 Accuracy: {best_trial.user_attrs['top1_acc']:.4f}")
print(f"Val Loss: {best_trial.user_attrs['val_loss']:.4f}")
print("Parametri:")
for k, v in best_trial.params.items():
    print(f"  {k}: {v}")

sorted_by_loss = sorted(study.trials, key=lambda t: t.user_attrs.get("val_loss", np.inf))
best_loss_trial = sorted_by_loss[0]
print("\n" + "="*60)
print("MIGLIOR MODELLO (per Val Loss minima)")
print("="*60)
print(f"Val Loss: {best_loss_trial.user_attrs['val_loss']:.4f}")
print(f"MRR: {best_loss_trial.user_attrs['mrr']:.4f}")
print(f"Top1 Accuracy: {best_loss_trial.user_attrs['top1_acc']:.4f}")
print("Parametri:")
for k, v in best_loss_trial.user_attrs['params'].items():
    print(f"  {k}: {v}")

sorted_by_top1 = sorted(study.trials, key=lambda t: t.user_attrs.get("top1_acc", -np.inf), reverse=True)
best_top1_trial = sorted_by_top1[0]
print("\n" + "="*60)
print("MIGLIOR MODELLO (per Top1 Accuracy)")
print("="*60)
print(f"Top1 Accuracy: {best_top1_trial.user_attrs['top1_acc']:.4f}")
print(f"MRR: {best_top1_trial.user_attrs['mrr']:.4f}")
print(f"Val Loss: {best_top1_trial.user_attrs['val_loss']:.4f}")
print("Parametri:")
for k, v in best_top1_trial.user_attrs['params'].items():
    print(f"  {k}: {v}")

print("\n" + "="*60)
print(f"TOP-{TOP_K} MODELLI SALVATI IN 'models_1408/'")
print("="*60)
saved_files = sorted([f for f in os.listdir("models_1408_6") if f.endswith(".pt")])
saved_trial_numbers = [int(f.split("_")[-1].split(".")[0]) for f in saved_files]
saved_trials = [t for t in study.trials if t.number in saved_trial_numbers]
saved_trials_sorted = sorted(saved_trials, key=lambda t: t.value, reverse=True)

for i, t in enumerate(saved_trials_sorted, 1):
    print(f"{i}. Trial {t.number}: MRR={t.value:.4f}, lr={t.params['lr']:.6f}")

print("\nBest model assoluto salvato in 'best_model.pt'")
print(f"Top-{TOP_K} modelli pronti per ensemble.")


[I 2025-11-15 19:36:50,390] A new study created in RDB with name: real_final_1408_topk
/tmp/ipykernel_39/3240220149.py:315: FutureWarning:

`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.



Epoch 1/33: Train=51.485764, Val=4.627232, LR=1.51e-04
  ✓ Best model saved (val_loss: 4.627232)
Epoch 2/33: Train=10.402566, Val=3.495568, LR=1.41e-04
  ✓ Best model saved (val_loss: 3.495568)
Epoch 3/33: Train=6.123573, Val=2.950746, LR=1.23e-04
  ✓ Best model saved (val_loss: 2.950746)
Epoch 4/33: Train=4.091044, Val=2.534994, LR=1.02e-04
  ✓ Best model saved (val_loss: 2.534994)
Epoch 5/33: Train=2.967992, Val=2.390106, LR=7.81e-05
  ✓ Best model saved (val_loss: 2.390106)
Epoch 6/33: Train=2.284444, Val=2.272287, LR=5.43e-05
  ✓ Best model saved (val_loss: 2.272287)
Epoch 7/33: Train=1.891454, Val=2.188343, LR=3.28e-05
  ✓ Best model saved (val_loss: 2.188343)
Epoch 8/33: Train=1.660134, Val=2.161234, LR=1.57e-05
  ✓ Best model saved (val_loss: 2.161234)
Epoch 9/33: Train=1.535524, Val=2.140497, LR=4.77e-06
  ✓ Best model saved (val_loss: 2.140497)
Epoch 10/33: Train=1.480232, Val=2.132761, LR=1.55e-04
  ✓ Best model saved (val_loss: 2.132761)
Epoch 11/33: Train=1.606062, Val=2.15

[I 2025-11-15 19:38:32,621] Trial 0 finished with value: 0.4443211328125 and parameters: {'lr': 0.00015524144639007706, 'temperature': 0.13999999999999999}. Best is trial 0 with value: 0.4443211328125.



📊 Risultati complessivi su 12500 campioni:
Top-1 accuracy : 19.79%
Top-5 accuracy : 93.38%
Top-10 accuracy: 98.14%
Rank medio     : 3.49
MRR            : 0.4443
Salvato nuovo modello TOP-K (trial 0, MRR=0.4443)
Salvato Best Model assoluto (MRR = 0.4443)
Epoch 1/33: Train=43.893641, Val=4.366695, LR=1.55e-04
  ✓ Best model saved (val_loss: 4.366695)
Epoch 2/33: Train=8.987441, Val=3.395024, LR=1.44e-04
  ✓ Best model saved (val_loss: 3.395024)
Epoch 3/33: Train=5.443299, Val=2.860514, LR=1.26e-04
  ✓ Best model saved (val_loss: 2.860514)
Epoch 4/33: Train=3.714510, Val=2.543327, LR=1.04e-04
  ✓ Best model saved (val_loss: 2.543327)
Epoch 5/33: Train=2.732141, Val=2.350167, LR=7.98e-05
  ✓ Best model saved (val_loss: 2.350167)
Epoch 6/33: Train=2.109783, Val=2.233591, LR=5.55e-05
  ✓ Best model saved (val_loss: 2.233591)
Epoch 7/33: Train=1.732010, Val=2.177421, LR=3.35e-05
  ✓ Best model saved (val_loss: 2.177421)
Epoch 8/33: Train=1.519348, Val=2.135212, LR=1.61e-05
  ✓ Best model sav

[I 2025-11-15 19:39:50,776] Trial 1 finished with value: 0.4443785546875 and parameters: {'lr': 0.0001586086639424876, 'temperature': 0.16}. Best is trial 1 with value: 0.4443785546875.


...


📊 Risultati complessivi su 12500 campioni:
Top-1 accuracy : 19.84%
Top-5 accuracy : 93.44%
Top-10 accuracy: 98.15%
Rank medio     : 3.49
MRR            : 0.4445
Trial 42 non entra nei Top-8 (MRR=0.4445 <= 0.4446)
Epoch 1/33: Train=55.611360, Val=5.116363, LR=1.53e-04
  ✓ Best model saved (val_loss: 5.116363)
Epoch 2/33: Train=11.544761, Val=3.600560, LR=1.42e-04
  ✓ Best model saved (val_loss: 3.600560)
Epoch 3/33: Train=6.685074, Val=3.065103, LR=1.24e-04
  ✓ Best model saved (val_loss: 3.065103)
Epoch 4/33: Train=4.400713, Val=2.661810, LR=1.03e-04
  ✓ Best model saved (val_loss: 2.661810)
Epoch 5/33: Train=3.103803, Val=2.416452, LR=7.87e-05
  ✓ Best model saved (val_loss: 2.416452)
Epoch 6/33: Train=2.376747, Val=2.281912, LR=5.47e-05
  ✓ Best model saved (val_loss: 2.281912)
Epoch 7/33: Train=1.950049, Val=2.217145, LR=3.30e-05
  ✓ Best model saved (val_loss: 2.217145)
Epoch 8/33: Train=1.689558, Val=2.167592, LR=1.58e-05
  ✓ Best model saved (val_loss: 2.167592)
Epoch 9/33: Trai

[I 2025-11-15 20:46:21,733] Trial 43 finished with value: 0.4441473828125 and parameters: {'lr': 0.00015646723760650528, 'temperature': 0.13}. Best is trial 22 with value: 0.44526328125.



📊 Risultati complessivi su 12500 campioni:
Top-1 accuracy : 19.79%
Top-5 accuracy : 93.36%
Top-10 accuracy: 98.10%
Rank medio     : 3.50
MRR            : 0.4441
Trial 43 non entra nei Top-8 (MRR=0.4441 <= 0.4446)
Epoch 1/33: Train=51.839844, Val=4.943997, LR=1.58e-04
  ✓ Best model saved (val_loss: 4.943997)
Epoch 2/33: Train=10.180227, Val=3.344074, LR=1.47e-04
  ✓ Best model saved (val_loss: 3.344074)
Epoch 3/33: Train=5.672800, Val=2.831144, LR=1.29e-04
  ✓ Best model saved (val_loss: 2.831144)
Epoch 4/33: Train=3.752183, Val=2.512261, LR=1.06e-04
  ✓ Best model saved (val_loss: 2.512261)
Epoch 5/33: Train=2.669813, Val=2.314416, LR=8.15e-05
  ✓ Best model saved (val_loss: 2.314416)
Epoch 6/33: Train=2.065837, Val=2.212172, LR=5.66e-05
  ✓ Best model saved (val_loss: 2.212172)
Epoch 7/33: Train=1.716901, Val=2.169519, LR=3.42e-05
  ✓ Best model saved (val_loss: 2.169519)
Epoch 8/33: Train=1.507170, Val=2.135800, LR=1.64e-05
  ✓ Best model saved (val_loss: 2.135800)
Epoch 9/33: Trai

[I 2025-11-15 20:47:51,770] Trial 44 finished with value: 0.4440584375 and parameters: {'lr': 0.00016206568863181845, 'temperature': 0.13999999999999999}. Best is trial 22 with value: 0.44526328125.



📊 Risultati complessivi su 12500 campioni:
Top-1 accuracy : 19.81%
Top-5 accuracy : 93.36%
Top-10 accuracy: 98.00%
Rank medio     : 3.51
MRR            : 0.4441
Trial 44 non entra nei Top-8 (MRR=0.4441 <= 0.4446)

MIGLIOR MODELLO (per MRR)
MRR: 0.4453
Top1 Accuracy: 0.1990
Val Loss: 2.0407
Parametri:
  lr: 0.00017136887779684436
  temperature: 0.13

MIGLIOR MODELLO (per Val Loss minima)
Val Loss: 2.0405
MRR: 0.4441
Top1 Accuracy: 0.1981
Parametri:
  hidden_dim: 1408
  lr: 0.00016206568863181845
  temperature: 0.13999999999999999

MIGLIOR MODELLO (per Top1 Accuracy)
Top1 Accuracy: 0.1990
MRR: 0.4452
Val Loss: 2.0416
Parametri:
  hidden_dim: 1408
  lr: 0.0001628478357865145
  temperature: 0.13

TOP-8 MODELLI SALVATI IN 'models_1408/'
1. Trial 22: MRR=0.4453, lr=0.000171
2. Trial 2: MRR=0.4452, lr=0.000163
3. Trial 35: MRR=0.4449, lr=0.000155
4. Trial 30: MRR=0.4448, lr=0.000161
5. Trial 27: MRR=0.4448, lr=0.000175
6. Trial 9: MRR=0.4448, lr=0.000170
7. Trial 3: MRR=0.4447, lr=0.000167
8

In [ ]:
# Load the best models for ensemble
models_1408 = []

MODEL_FOLDER = r"\Models\Best_submission"

saved_files = sorted([f for f in os.listdir(MODEL_FOLDER) if f.endswith(".pt")])

for fname in saved_files:
    model = ExpandCompressMLP(
        input_dim=1024,
        hidden_dim=1408,
        bottleneck_dim=1024,
        output_dim=1536,
        dropout=0.35,
        use_residual=False
    ).to(device)

    model.load_state_dict(torch.load(os.path.join(MODEL_FOLDER, fname), map_location=device))
    model.eval()
    models_1408.append(model)
    print(f"Caricato {fname}")


# Creation of weights for ensemble based on MRR


mrr_weights = [0.4452, 0.4447, 0.4448, 0.4453, 0.4448, 0.4448, 0.4446, 0.4449]
mrr_weights = [w / sum(mrr_weights) for w in mrr_weights]

print("Pesi MRR:", [f"{w:.4f}" for w in mrr_weights])

Caricato model_1408_trial_2.pt
Caricato model_1408_trial_22.pt
Caricato model_1408_trial_27.pt
Caricato model_1408_trial_3.pt
Caricato model_1408_trial_30.pt
Caricato model_1408_trial_31.pt
Caricato model_1408_trial_35.pt
Caricato model_1408_trial_9.pt
Pesi MRR: ['0.1251', '0.1249', '0.1250', '0.1251', '0.1250', '0.1250', '0.1249', '0.1250']


In [8]:
ensemble_model_weighted = EnsembleModel(models_1408, weights=mrr_weights).to(device)

In [10]:
res = evaluate_model(ensemble_model_weighted, val_loader, device)


📊 Risultati complessivi su 12500 campioni:
Top-1 accuracy : 20.00%
Top-5 accuracy : 94.22%
Top-10 accuracy: 98.46%
Rank medio     : 3.42
MRR            : 0.4473


In [ ]:
test_data = load_data(r'\test\test\test.clean.npz')
test_embds = torch.from_numpy(test_data['captions/embeddings']).float()
caption_ids = test_data['captions/ids']

pred_embds = predict_in_batches(ensemble_model_weighted, test_embds, device, batch_size=64)

submission = generate_submission(caption_ids, pred_embds, 'submission_ensambling_speriamo_sono_le_ultime_speranze_pt_6.csv') # Qui era finita del tutto la fantasia

Generating submission file...
✓ Saved submission to submission_ensambling_speriamo_sono_le_ultime_speranze_pt_6_only_5.csv
